## Combine all datasets and Train Test Spilt

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import sys
import numpy as np
sys.path.append('../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

np.random.seed(42)  # For reproducibility


## 1. Combine all datasets

In [43]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../data_1s_30hz_7class/curb_1s_combined_all.pkl', 'rb') as f:
    curb_data = pickle.load(f)
    data_curb_0 = curb_data['scene_0']
    data_curb_1 = curb_data['scene_1']
    
# Load the other surface types (these appear to be stored as arrays)
with open('../data_1s_30hz_7class/asphalt_1s_combined_all.pkl', 'rb') as f:
    data_asphalt = pickle.load(f)

with open('../data_1s_30hz_7class/cobblestone_1s_combined_all.pkl', 'rb') as f:
    data_cobblestone = pickle.load(f)

with open('../data_1s_30hz_7class/compactgravel_1s_combined_all.pkl', 'rb') as f:
    data_compact_gravel = pickle.load(f)

with open('../data_1s_30hz_7class/dirt_1s_combined_all.pkl', 'rb') as f:
    data_Dirt = pickle.load(f)

with open('../data_1s_30hz_7class/pavingstone_1s_combined_all.pkl', 'rb') as f:
    data_PavingStone = pickle.load(f)

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)
print("Asphalt:", data_asphalt.shape)
print("Cobblestone:", data_cobblestone.shape)
print("Compact Gravel:", data_compact_gravel.shape)
print("Dirt:", data_Dirt.shape)
print("Paving Stone:", data_PavingStone.shape)

Curb (scene 0): (617, 30, 3)
Curb (scene 1): (617, 30, 3)
Asphalt: (704, 30, 3)
Cobblestone: (486, 30, 3)
Compact Gravel: (479, 30, 3)
Dirt: (577, 30, 3)
Paving Stone: (645, 30, 3)


In [28]:
# Create binary class dataset: curb_1 vs. random even samples from other classes
# First, separate curb_1 data
curb_1_data = [(segment, "curb_1") for segment in data_curb_1]
print(f"Total curb_1 samples: {len(curb_1_data)}")

# Get count of curb_1 samples to know how many we need from other classes
curb_1_count = len(curb_1_data)

# Create list of other datasets
other_datasets = [
    (data_curb_0, "curb_0"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone")
]

# Calculate how many samples to take from each other class for even distribution
samples_per_class = curb_1_count // len(other_datasets)
print(f"Taking {samples_per_class} samples from each of the other 6 classes")

# Randomly select samples from other classes
other_class_data = []
for data, label in other_datasets:
    # Randomly select indices
    selected_indices = np.random.choice(len(data), samples_per_class, replace=False)
    # Add selected samples to other_class_data
    for idx in selected_indices:
        other_class_data.append((data[idx], "non_curb"))  # Label all other classes as "non_curb"

print(f"Total non_curb samples: {len(other_class_data)}")

# Combine curb_1 and other class data
combined_dataset = other_class_data + curb_1_data
print(f"Total binary dataset samples: {len(combined_dataset)}")



Total curb_1 samples: 617
Taking 102 samples from each of the other 6 classes
Total non_curb samples: 612
Total binary dataset samples: 1229


In [29]:
# Save the binary dataset
with open('../data_1s_30hz_2class/binary_dataset.pkl', 'wb') as f:
    pickle.dump(combined_dataset, f)

## 2. Train Test Spilt


In [30]:
# 80% train, 20% test
train_set, test_set = train_test_split(combined_dataset, test_size=0.2, random_state=42, shuffle=True, stratify=[label for data, label in combined_dataset])
print(len(train_set), len(test_set))
# Separate sensor values and labels
X_train = [x for x, _ in train_set]
y_train = [label for _, label in train_set]
X_test = [x for x, _ in test_set]
y_test = [label for _, label in test_set]
print(f"Label train: {len(y_train)}, Label test: {len(y_test)}")

983 246
Label train: 983, Label test: 246


## 3. Normalise dataset

In [31]:
X_train_normalized = normalize_3d_data(X_train)

# 4. Labels from string to integer

In [36]:
# Define custom mapping: curb_1=1, non_curb=0
custom_mapping = {"curb_1": 1, "non_curb": 0}

# Apply the custom mapping
y_train_int = np.array([custom_mapping[label] for label in y_train])
y_test_int = np.array([custom_mapping[label] for label in y_test])

# Create and manually adjust the label_encoder to match your encoding
label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(["non_curb", "curb_1"])  # Ensures 0=non_curb, 1=curb_1

print("Classes:", label_encoder.classes_)
print("First 10 y_train_int:", y_train_int[:10])
print("First 10 y_test_int:", y_test_int[:10])
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

Classes: ['non_curb' 'curb_1']
First 10 y_train_int: [1 1 0 1 1 1 1 0 1 0]
First 10 y_test_int: [1 0 1 1 1 1 0 0 0 0]
0: non_curb
1: curb_1


## 5: One-hot encode the labels

In [37]:
y_train_onehot = to_categorical(y_train_int)
y_test_onehot = to_categorical(y_test_int)

print(y_train_onehot.shape)
print(y_test_onehot.shape)

(983, 2)
(246, 2)


In [38]:
# Randomly select an index and check that the one-hot encoding matches the original label
r = np.random.randint(len(y_train_int))
assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_test_int))
assert y_test_onehot[r].argmax() == y_test_int[r]

## 6. Save train, test data and labels

In [39]:
# Save test data
with open('../data_1s_30hz_2class/X_test_data.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open('../data_1s_30hz_2class/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [40]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (786, 30, 3) (786, 2)
Validation shape: (197, 30, 3) (197, 2)


In [41]:
# Save training and validation data
with open('../data_1s_30hz_2class/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../data_1s_30hz_2class/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../data_1s_30hz_2class/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../data_1s_30hz_2class/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)